# A/B Test: Query Rewrite & HyDE vs Original

Users sometimes ask vague questions (e.g., "Which professor teaches AI related courses?" vs "Who teaches generative AI?") that retrieve different chunks and produce inconsistent answers.

This notebook evaluates two LLM-based approaches to improve retrieval for vague queries:
- **A (Original):** Use user query directly for retrieval (current production)
- **B (Query Rewrite):** LLM rewrites query into keyword-rich form before retrieval
- **C (HyDE):** LLM generates a hypothetical answer, embed that for semantic search, keep original BM25

**Conclusion:** Both B and C underperform the original pipeline. See the README for analysis.

## Setup

In [1]:
import sys, os, json
import numpy as np
from collections import Counter

BACKEND_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(BACKEND_DIR, "data")):
    BACKEND_DIR = "/Users/pengxiao/Downloads/UChicago_ADS_RAG/backend"

os.chdir(BACKEND_DIR)
sys.path.insert(0, BACKEND_DIR)
print(f"Working directory: {os.getcwd()}")

from dotenv import load_dotenv
load_dotenv(os.path.join(BACKEND_DIR, ".env"))

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from langchain_google_genai import ChatGoogleGenerativeAI

from retrieval import (
    retrieve, rerank, tokenize_for_bm25, expand_query, _get_url, _is_dup
)

# Load data
print("Loading chunks...")
with open("data/chunked_documents_dedup.json") as f:
    chunk_records = json.load(f)
print(f"  {len(chunk_records)} chunks loaded")

print("Loading embedder...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Loading FAISS index...")
faiss_index = faiss.read_index("data/uchicago_ads_faiss_dedup.index")

print("Building BM25 index...")
tokenized = [tokenize_for_bm25(c["text"]) for c in chunk_records]
bm25 = BM25Okapi(tokenized)

print("Loading cross-encoder...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# LLM for query rewriting / HyDE
rewrite_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.0)

print("\u2705 All resources loaded.")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Working directory: /Users/pengxiao/Downloads/UChicago_ADS_RAG/backend
Loading chunks...
  1012 chunks loaded
Loading embedder...
Loading FAISS index...
Building BM25 index...
Loading cross-encoder...
✅ All resources loaded.


## Helper Functions

In [2]:
async def rewrite_query(query: str) -> str:
    """Rewrite into alternative search query."""
    response = await rewrite_llm.ainvoke(
        "You are a search query optimizer for UChicago's MS in Applied Data Science program website. "
        "Rewrite the following question into a short, alternative search query that uses different "
        "keywords but preserves the same intent. Keep it concise (under 15 words). "
        "Return ONLY the rewritten query, nothing else.\n\n"
        f"Question: {query}"
    )
    return response.content.strip()


async def hyde_generate(query: str) -> str:
    """Generate a hypothetical answer for HyDE."""
    response = await rewrite_llm.ainvoke(
        "You are a knowledgeable advisor for UChicago's MS in Applied Data Science program. "
        "Answer the following question in 2-3 sentences as if you were quoting from the program website. "
        "Be specific and factual. Return ONLY the answer, nothing else.\n\n"
        f"Question: {query}"
    )
    return response.content.strip()


def hyde_retrieve(original_q, hyde_doc, top_k=15):
    """HyDE retrieval: semantic search with hypothetical doc embedding, BM25 with original query."""
    hyde_emb = embedder.encode([hyde_doc], normalize_embeddings=True)[0].astype("float32")
    n = len(chunk_records)

    # Semantic ranking from HyDE embedding
    hyde_emb_q = hyde_emb.reshape(1, -1)
    sem_order = faiss_index.search(hyde_emb_q, n)[1][0]
    sem_rank = np.empty(n, dtype=int)
    for rank, idx in enumerate(sem_order):
        sem_rank[idx] = rank

    # BM25 ranking from ORIGINAL query
    expanded = expand_query(original_q)
    bm25_tokens = tokenize_for_bm25(expanded)
    bm25_scores = bm25.get_scores(bm25_tokens)
    bm25_order = np.argsort(-bm25_scores)
    bm25_rank = np.empty(n, dtype=int)
    for rank, idx in enumerate(bm25_order):
        bm25_rank[idx] = rank

    # RRF fusion
    k_rrf = 60
    rrf = np.zeros(n)
    for i in range(n):
        rrf[i] += 1.0 / (k_rrf + sem_rank[i] + 1)
        rrf[i] += 1.0 / (k_rrf + bm25_rank[i] + 1)

    # Dedup + top-k (match retrieve() output format)
    order = np.argsort(-rrf)
    candidates = []
    seen_texts = []
    for idx in order:
        if len(candidates) >= top_k:
            break
        c = chunk_records[idx]
        if any(_is_dup(c["text"], s) for s in seen_texts):
            continue
        meta = c.get("metadata", {})
        candidates.append({
            "chunk_id": c["chunk_id"],
            "text": c["text"],
            "url": _get_url(meta),
            "page_title": meta.get("page_title", ""),
            "labels": meta.get("labels", []),
            "score": float(rrf[idx]),
        })
        seen_texts.append(c["text"])
    return candidates


def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    return sum(1 for rid in retrieved_ids[:k] if rid in relevant_ids) / len(relevant_ids)


def reciprocal_rank(retrieved_ids, relevant_ids):
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

## Qualitative Comparison (Sample Queries)

Three semantically similar queries that produce inconsistent answers in the current system.

In [3]:
TEST_QUERIES = [
    "Which professor teaches AI related courses?",
    "Who teaches generative AI?",
    "Which professor teaches generative AI?",
]

print("=" * 90)
print("A/B/C TEST: Original vs Rewrite vs HyDE")
print("=" * 90)

for q in TEST_QUERIES:
    rewritten = await rewrite_query(q)
    hyde_doc = await hyde_generate(q)

    # A: Original
    cands_a = retrieve(q, chunk_records, embedder, bm25, faiss_index, top_k=15)
    hits_a = rerank(q, cands_a, cross_encoder, top_k=5)

    # B: Rewrite (replace)
    cands_b = retrieve(rewritten, chunk_records, embedder, bm25, faiss_index, top_k=15)
    hits_b = rerank(rewritten, cands_b, cross_encoder, top_k=5)

    # C: HyDE (semantic from hypothetical doc, BM25 from original, rerank with original)
    cands_c = hyde_retrieve(q, hyde_doc, top_k=15)
    hits_c = rerank(q, cands_c, cross_encoder, top_k=5)

    print(f"\n{'\u2500' * 90}")
    print(f"Query:    {q}")
    print(f"Rewrite:  {rewritten}")
    print(f"HyDE doc: {hyde_doc[:120]}...")

    print(f"\n  {'[A] Original':<30} {'[B] Rewrite':<30} {'[C] HyDE':<30}")
    print(f"  {'\u2500' * 30} {'\u2500' * 30} {'\u2500' * 30}")
    for i in range(5):
        a_t = hits_a[i]["page_title"][:27] if i < len(hits_a) else ""
        b_t = hits_b[i]["page_title"][:27] if i < len(hits_b) else ""
        c_t = hits_c[i]["page_title"][:27] if i < len(hits_c) else ""
        print(f"  {i+1}. {a_t:<28} {b_t:<28} {c_t:<28}")

A/B/C TEST: Original vs Rewrite vs HyDE

──────────────────────────────────────────────────────────────────────────────────────────
Query:    Which professor teaches AI related courses?
Rewrite:  AI course instructors
HyDE doc: "Our curriculum features AI-focused courses taught by faculty with extensive research experience. For instance, Professo...

  [A] Original                   [B] Rewrite                    [C] HyDE                      
  ────────────────────────────── ────────────────────────────── ──────────────────────────────
  1. Affiliated Faculty | DSI     Faculty, Instructors, Staff  Faculty, Instructors, Staff 
  2. Affiliated Faculty | DSI     Batu Gundogdu, PhD | DSI     Faculty, Instructors, Staff 
  3. Affiliated Faculty | DSI     AI Initiative Shares UChica  Batu Gundogdu, PhD | DSI    
  4. Online Program | DSI         In-Person Program | DSI      Utku Pamuksuz, PhD | DSI    
  5. Course Progressions | DSI    Course Progressions | DSI    Batu Gundogdu, PhD | DSI  

## Quantitative Comparison (Golden Test Set)

Run all three strategies on the full 55-query golden test set and compare Recall@5 and MRR.

In [4]:
with open("data/golden_test_set.json") as f:
    golden = json.load(f)

metrics = {s: {"recall@5": [], "mrr": []} for s in ["A_original", "B_rewrite", "C_hyde"]}

_TRANSLATIONS = {
    "多少钱": "How much does the program cost?",
    "这个项目要读多久": "How long does the program take to complete?",
    "怎么申请": "How do I apply to the program?",
    "有奖学金吗": "Are there scholarships available?",
    "毕业后能找到什么工作": "What jobs can graduates find after the program?",
    "线上和线下有什么区别": "What is the difference between online and in-person programs?",
}

print("Running golden test set A/B/C comparison...")
print(f"  A: Original query        B: LLM Rewrite        C: HyDE")
print(f"\n{'Query':<45} {'R@5(A)':>7} {'R@5(B)':>7} {'R@5(C)':>7}")
print("\u2500" * 75)

for item in golden:
    q = item["query"]
    relevant = set(item["relevant_chunk_ids"])

    if not all(ord(c) < 128 for c in q if not c.isspace()):
        en_q = _TRANSLATIONS.get(q, q)
    else:
        en_q = q

    rewritten = await rewrite_query(en_q)
    hyde_doc = await hyde_generate(en_q)

    # A: Original
    cands_a = retrieve(en_q, chunk_records, embedder, bm25, faiss_index, top_k=15)
    hits_a = rerank(en_q, cands_a, cross_encoder, top_k=5)
    ids_a = [h["chunk_id"] for h in hits_a]

    # B: Rewrite
    cands_b = retrieve(rewritten, chunk_records, embedder, bm25, faiss_index, top_k=15)
    hits_b = rerank(rewritten, cands_b, cross_encoder, top_k=5)
    ids_b = [h["chunk_id"] for h in hits_b]

    # C: HyDE
    cands_c = hyde_retrieve(en_q, hyde_doc, top_k=15)
    hits_c = rerank(en_q, cands_c, cross_encoder, top_k=5)
    ids_c = [h["chunk_id"] for h in hits_c]

    for label, ids in [("A_original", ids_a), ("B_rewrite", ids_b), ("C_hyde", ids_c)]:
        metrics[label]["recall@5"].append(recall_at_k(ids, relevant, 5))
        metrics[label]["mrr"].append(reciprocal_rank(ids, relevant))

    r5_a = metrics["A_original"]["recall@5"][-1]
    r5_b = metrics["B_rewrite"]["recall@5"][-1]
    r5_c = metrics["C_hyde"]["recall@5"][-1]
    print(f"  {q[:43]:<45} {r5_a:>5.2f}   {r5_b:>5.2f}   {r5_c:>5.2f}")

avg = lambda lst: sum(lst) / len(lst) if lst else 0
print(f"\n{'=' * 75}")
print(f"AGGREGATE ({len(golden)} queries)")
print(f"  {'Metric':<12} {'A(orig)':>10} {'B(rewrite)':>10} {'C(HyDE)':>10} {'B-A':>8} {'C-A':>8}")
print(f"  {'\u2500'*12} {'\u2500'*10} {'\u2500'*10} {'\u2500'*10} {'\u2500'*8} {'\u2500'*8}")
for key in ["recall@5", "mrr"]:
    a = avg(metrics["A_original"][key])
    b = avg(metrics["B_rewrite"][key])
    c = avg(metrics["C_hyde"][key])
    print(f"  {key:<12} {a:>10.3f} {b:>10.3f} {c:>10.3f} {b-a:>+8.3f} {c-a:>+8.3f}")

Running golden test set A/B/C comparison...
  A: Original query        B: LLM Rewrite        C: HyDE

Query                                          R@5(A)  R@5(B)  R@5(C)
───────────────────────────────────────────────────────────────────────────
  What are the core courses in the MS in Appl    0.33    0.33    0.00
  What is the tuition cost for the program?      1.00    0.50    1.00
  What scholarships are available for the pro    1.00    0.50    1.00
  What are the admission requirements for the    0.38    0.00    0.12
  Tell me about the capstone project.            1.00    0.33    1.00
  Is the GRE or GMAT required?                   1.00    0.33    1.00
  When will I receive my admission decision?     1.00    1.00    1.00
  What are the minimum TOEFL and IELTS scores    1.00    1.00    1.00
  What is the difference between the online a    0.60    0.40    0.80
  How long does it take to complete the progr    1.00    0.00    0.00
  What is the sample full-time schedule?         1.0